# Comprehensive Quantization Comparison

Compares accuracy, latency (averaged over 30 runs), effective FLOPs, and theoretical model size across:
- **Baseline (FP32)**
- **Static INT8** (PyTorch native)
- **Static INT6** (PTQ Simulation)
- **Static INT4** (PTQ Simulation)
- **DQNet** (Per-layer dynamic routing)
- **Global DQNet** (Experiment 1: INT5 vs INT7 image-level routing)

In [ ]:
!git clone https://github.com/MrRoboto11102003/Quantization_project.git 2>/dev/null; true
!pip install thop -q
import sys
sys.path.append('Quantization_project')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.quantization as tq
import copy, time
import numpy as np
import matplotlib.pyplot as plt
from thop import profile

import torchvision
import torchvision.transforms as transforms

from resnet import resnet20, BasicBlock
from DQ_resnet import dq_resnet20, compute_bitops, max_bitops, get_mean_bits, BIT_OPTIONS
from experiment1 import global_dq_resnet20, compute_global_bitops

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

calibration_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=transform_test)
calibration_loader = torch.utils.data.DataLoader(calibration_set, batch_size=128, shuffle=False, num_workers=2)

## Evaluation Helpers

In [ ]:
def evaluate(model, loader, device='cpu'):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            out = model(inputs)
            if isinstance(out, tuple):
                out = out[0]
            _, predicted = out.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return 100. * correct / total

def measure_latency(model, loader, device='cpu', runs=30):
    model.eval()
    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            out = model(inputs)
        if torch.cuda.is_available() and device != 'cpu': torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(runs):
            if torch.cuda.is_available() and device != 'cpu': torch.cuda.synchronize()
            start = time.time()
            for inputs, _ in loader:
                inputs = inputs.to(device)
                _ = model(inputs)
            if torch.cuda.is_available() and device != 'cpu': torch.cuda.synchronize()
            times.append(time.time() - start)
    return np.mean(times), np.std(times)

def calc_size_mb(model, effective_bits=32):
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return (total_params * effective_bits) / (8 * 1024 * 1024)

results = {}

## 1 — Baseline FP32

In [ ]:
BASELINE_PATH = 'Quantization_project/resnet20_base_best.pth'
net_base = resnet20()
net_base.load_state_dict(torch.load(BASELINE_PATH, map_location='cpu'))
net_base = net_base.to(device)
net_base.eval()

base_acc = evaluate(net_base, testloader, device)
base_lat, base_std = measure_latency(net_base, testloader, device)
base_flops, _ = profile(net_base, inputs=(torch.randn(1, 3, 32, 32).to(device),), verbose=False)
base_size = calc_size_mb(net_base, 32)

results['Baseline FP32'] = {'acc': base_acc, 'lat': base_lat, 'std': base_std, 'flops': base_flops, 'size': base_size}
print(f'Baseline — Acc: {base_acc:.2f}% | Lat: {base_lat:.4f}s | FLOPs: {base_flops/1e6:.2f}M | Size: {base_size:.2f}MB')

## 2 — Static INT8 Quantization (PyTorch Native CPU)

In [ ]:
class QuantizedBasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False), nn.BatchNorm2d(planes))
        self.skip_add = torch.nn.quantized.FloatFunctional()
        self.relu1 = nn.ReLU()
        self.relu2 = nn.ReLU()
    def forward(self, x):
        out = self.relu1(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu2(self.skip_add.add(out, self.shortcut(x)))

class QuantizedResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super().__init__()
        self.in_planes = 16
        self.quant = torch.quantization.QuantStub()
        self.dequant = torch.quantization.DeQuantStub()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU()
        self.layer1 = self._make_layer(block, 16, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 32, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 64, num_blocks[2], stride=2)
        self.linear = nn.Linear(64, num_classes)
    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(self.quant(x))))
        out = self.layer3(self.layer2(self.layer1(out)))
        out = F.avg_pool2d(out, out.size()[3])
        out = self.dequant(self.linear(out.view(out.size(0), -1)))
        return out

net_int8 = QuantizedResNet(QuantizedBasicBlock, [3, 3, 3])
net_int8.load_state_dict(torch.load(BASELINE_PATH, map_location='cpu'), strict=False)
net_int8.eval()
net_int8.qconfig = tq.get_default_qconfig('fbgemm')
net_int8_prepared = tq.prepare(net_int8, inplace=False)

with torch.no_grad():
    for i, (inputs, _) in enumerate(calibration_loader):
        net_int8_prepared(inputs)
        if i >= 50: break

net_int8_q = tq.convert(net_int8_prepared, inplace=False)
int8_acc = evaluate(net_int8_q, testloader, 'cpu')
int8_lat, int8_std = measure_latency(net_int8_q, testloader, 'cpu')
results['Static INT8'] = {'acc': int8_acc, 'lat': int8_lat, 'std': int8_std, 'flops': base_flops/4, 'size': calc_size_mb(net_base, 8)}
print(f'Static INT8 — Acc: {int8_acc:.2f}% | Lat: {int8_lat:.4f}s')

## 3 & 4 — Static INT6 and INT4 (PTQ Folded Simulation on GPU)
PyTorch's native static engine struggles below INT8. We use Post-Training Quantization simulation (BN Folding + FakeQuant) to evaluate INT6 and INT4.

In [ ]:
def fold_bn(conv, bn):
    w = conv.weight.data.clone()
    b = torch.zeros(w.size(0), device=w.device)
    gamma, beta = bn.weight.data, bn.bias.data
    mean, var, eps = bn.running_mean, bn.running_var, bn.eps
    std = torch.sqrt(var + eps)
    conv.weight.data = w * (gamma / std).view(-1, 1, 1, 1)
    conv.bias = nn.Parameter(gamma * (b - mean) / std + beta)
    bn.weight.data.fill_(1.0)
    bn.bias.data.fill_(0.0)
    bn.running_mean.fill_(0.0)
    bn.running_var.fill_(1.0)

def quantize_tensor(t, num_bits):
    qmin, qmax = -(2**(num_bits-1)), 2**(num_bits-1) - 1
    min_val, max_val = t.min(), t.max()
    scale = (max_val - min_val) / (qmax - qmin)
    scale = max(scale.item(), 1e-8)
    zero_point = qmin - round(min_val.item() / scale)
    t_q = torch.clamp(torch.round(t / scale + zero_point), qmin, qmax)
    return (t_q - zero_point) * scale

def get_ptq_model(bits):
    model_q = copy.deepcopy(net_base).to(device)
    model_q.eval()
    fold_bn(model_q.conv1, model_q.bn1)
    for layer in [model_q.layer1, model_q.layer2, model_q.layer3]:
        for block in layer:
            fold_bn(block.conv1, block.bn1)
            fold_bn(block.conv2, block.bn2)
            if len(block.shortcut) > 0:
                fold_bn(block.shortcut[0], block.shortcut[1])
    
    for name, param in model_q.named_parameters():
        if 'weight' in name and param.dim() > 1:
            param.data = quantize_tensor(param.data, bits)
    return model_q

# Evaluate INT6
net_int6 = get_ptq_model(6)
int6_acc = evaluate(net_int6, testloader, device)
int6_lat, int6_std = measure_latency(net_int6, testloader, device)
results['Static INT6'] = {'acc': int6_acc, 'lat': int6_lat, 'std': int6_std, 'flops': base_flops*(6/32), 'size': calc_size_mb(net_base, 6)}
print(f'Static INT6 — Acc: {int6_acc:.2f}%')

# Evaluate INT4
net_int4 = get_ptq_model(4)
int4_acc = evaluate(net_int4, testloader, device)
int4_lat, int4_std = measure_latency(net_int4, testloader, device)
results['Static INT4'] = {'acc': int4_acc, 'lat': int4_lat, 'std': int4_std, 'flops': base_flops*(4/32), 'size': calc_size_mb(net_base, 4)}
print(f'Static INT4 — Acc: {int4_acc:.2f}%')

## 5 — DQNet (Dynamic Per-Layer Bit Controller)

In [ ]:
DQ_PATH = 'Quantization_project/dq_resnet20 (2).pth'
dq_model = dq_resnet20(bit_options=BIT_OPTIONS).to(device)
dq_model.load_state_dict(torch.load(DQ_PATH, map_location=device))
dq_model.eval()

dq_acc = evaluate(dq_model, testloader, device)
dq_lat, dq_std = measure_latency(dq_model, testloader, device)

fp32_max = max_bitops(dq_model.layer_flops)
bitops_ratios, all_bits = [], []
with torch.no_grad():
    for inputs, _ in testloader:
        inputs = inputs.to(device)
        _, soft_bits_list = dq_model(inputs)
        bitops_ratios.append(compute_bitops(soft_bits_list, dq_model.layer_flops).item() / fp32_max)
        all_bits.append(get_mean_bits(soft_bits_list).cpu().numpy())

avg_bitops_ratio = np.mean(bitops_ratios)
avg_bits = np.concatenate(all_bits).mean()

results['DQNet'] = {'acc': dq_acc, 'lat': dq_lat, 'std': dq_std, 'flops': base_flops * avg_bitops_ratio, 'size': calc_size_mb(dq_model, avg_bits)}
print(f'DQNet — Acc: {dq_acc:.2f}% | Lat: {dq_lat:.4f}s | Avg Bits: {avg_bits:.2f}')

## 6 — Global DQNet (INT5 vs INT7 Dynamic Batch Routing)

In [ ]:
GLOBAL_DQ_PATH = 'Quantization_project/global_dq_resnet20 (1).pth'
try:
    global_model = global_dq_resnet20().to(device)
    global_model.load_state_dict(torch.load(GLOBAL_DQ_PATH, map_location=device))
    global_model.eval()

    gl_acc = evaluate(global_model, testloader, device)
    gl_lat, gl_std = measure_latency(global_model, testloader, device)

    int5_c, int7_c, total_c = 0, 0, 0
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs = inputs.to(device)
            _, hard_idx = global_model(inputs)
            int5_c += (hard_idx == 0).sum().item()
            int7_c += (hard_idx == 1).sum().item()
            total_c += inputs.size(0)

    gl_avg_bits = (int5_c * 5 + int7_c * 7) / total_c
    gl_flops_ratio = ((int5_c * 5*5) + (int7_c * 7*7)) / (total_c * 32*32)

    results['Global DQNet (Exp 1)'] = {'acc': gl_acc, 'lat': gl_lat, 'std': gl_std, 'flops': base_flops * gl_flops_ratio, 'size': calc_size_mb(global_model, gl_avg_bits)}
    print(f'Global DQNet — Acc: {gl_acc:.2f}% | Lat: {gl_lat:.4f}s | Avg Bits: {gl_avg_bits:.2f} (INT5: {int5_c/total_c*100:.1f}%)')
except FileNotFoundError:
    print("global_dq_resnet20 (1).pth not found! Please train experiment 1 first.")

## Final Summary & Visualizations

In [ ]:
print(f'{"Model":<22} | {"Accuracy (%)":>12} | {"Latency (s)":>12} | {"FLOPs (M)":>10} | {"Size (MB)":>10}')
print('-' * 76)
for name, r in results.items():
    print(f'{name:<22} | {r["acc"]:>11.2f}% | {r["lat"]:>10.4f}s | {r["flops"]/1e6:>9.2f}M | {r["size"]:>8.2f}MB')

In [ ]:
names = list(results.keys())
accs  = [results[n]['acc']  for n in names]
lats  = [results[n]['lat']  for n in names]
stds  = [results[n]['std']  for n in names]
flops = [results[n]['flops'] / 1e6 for n in names]
sizes = [results[n]['size'] for n in names]

colors = plt.cm.get_cmap('Set2')(np.linspace(0, 1, len(names)))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

# Accuracy
bars = axes[0].bar(names, accs, color=colors)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Test Accuracy')
axes[0].set_ylim(min(accs) - 5, max(accs) + 2)
for bar, v in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

# Latency
bars = axes[1].bar(names, lats, yerr=stds, color=colors, capsize=5)
axes[1].set_ylabel('Inference Time (s)')
axes[1].set_title('Avg Inference Latency')
for bar, v in zip(bars, lats):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + max(stds)*0.2, f'{v:.3f}s', ha='center', fontweight='bold')

# FLOPs
bars = axes[2].bar(names, flops, color=colors)
axes[2].set_ylabel('FLOPs (M)')
axes[2].set_title('Effective FLOPs')
for bar, v in zip(bars, flops):
    axes[2].text(bar.get_x() + bar.get_width()/2, v + max(flops)*0.02, f'{v:.1f}M', ha='center', fontweight='bold')

# Size
bars = axes[3].bar(names, sizes, color=colors)
axes[3].set_ylabel('Size (MB)')
axes[3].set_title('Theoretical Model Footprint (MB)')
for bar, v in zip(bars, sizes):
    axes[3].text(bar.get_x() + bar.get_width()/2, v + max(sizes)*0.02, f'{v:.2f}', ha='center', fontweight='bold')

for ax in axes:
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('comparison_full_results.png', dpi=150, bbox_inches='tight')
plt.show()